In [0]:
%sql
CREATE OR REPLACE VIEW cobranzas.semantic.vw_rendimiento_agentes AS

WITH resumen_promesas AS (
    SELECT 
        R.gestor,
        COUNT_IF(P.estado_promesa_id = 4) AS cumplidas,
        COUNT_IF(P.estado_promesa_id = 1) AS incumplidas,
        COUNT_IF(P.estado_promesa_id = 2) AS postergadas,
        COUNT_IF(P.estado_promesa_id = 3) AS pendientes,
        COUNT(*) AS total_promesas
    FROM cobranzas.gold.fact_promesas P
    INNER JOIN cobranzas.gold.fact_cuenta C
        ON P.cuenta_id = C.cuenta_id
    INNER JOIN cobranzas.gold.dim_responsables_cuenta R
        ON C.responsables_cuenta_id = R.responsables_cuenta_id
    WHERE P.estado_promesa_id IN (1, 2, 3, 4)
    GROUP BY R.gestor
)

SELECT 
    gestor,
    cumplidas,
    incumplidas,
    postergadas,
    pendientes,
    total_promesas,
    
    ROUND((cumplidas * 100.0) / NULLIF(total_promesas, 0), 2) AS Efectividad,
    ROUND(((total_promesas - pendientes) * 100.0) / NULLIF(total_promesas, 0), 2) AS Avance_Recuperaciones,
    ROUND((incumplidas * 100.0) / NULLIF((cumplidas + incumplidas), 0), 2) AS Promesas_Rotas,
    ROUND((postergadas * 100.0) / NULLIF(total_promesas, 0), 2) AS Postergaciones

FROM resumen_promesas;